## Example.ipynb

This is a short example notebook to demonstrate how to implement the pseudo code in practice. Here I am using numpy simulations that take as input a protocol, and the number of iterations and return the counts of each measurement outcome.

### Import Libraries

In [2]:
import numpy as np
import helper_functions.RLA_functions as rla
import helper_functions.numpy_simulation as sim
import helper_functions.numpy_protocols as pmp

import matplotlib.pyplot as plt
import csv
from datetime import datetime
from pathlib import Path

### Initialize simulation parameters

In [10]:
# Network Parameters
actual_thetas = [0.3,0.01,0.15]

# Estimator Parameters
iterations = 100 # Pick 100 protocols 
trials_per_iteration = 1000 # Run each protocol 1000 times

initial_guess_thetas = [0.2, 0.2, 0.2] # Could be anything

# Load in the RLA models
models = rla.get_committee_models("Models/bit_flip_committee_1")

### Perform Simulation

In [11]:
## Create Results lists
GUESSES = [initial_guess_thetas]
PROTOCOLS = []
PROPORTIONS = []
TRIALS = []

for i in range(iterations):
        if i % 10 == 0: # Helpful logging during simulation ~ optional
            print(f"Iteration {i}/{iterations}")

        # Get current guess
        guess=GUESSES[-1]

        # Get optimal protocol for current guess
        _,protocol = rla.committee_predict_D(models=models, thetas=guess, is_OD=False) # not Using obj immplementation

        # Run simulation using the optimal protocol and actual thetas to get measurement proportions
            # This line should be replaced with your own netSquid Simulation, if you want you could define it
            # as a function in a different module or you could just copy past the code here.
        results = sim.simulate_bit_flip(thetas=actual_thetas, protocol=protocol, trials=trials_per_iteration)

        
        # Update arrays
        PROTOCOLS.append(protocol)
        PROPORTIONS.append(results)
        TRIALS.append(trials_per_iteration)
        

        # Plug into estimation function 
        new_guess = rla.estimator_D(all_protocols=PROTOCOLS, 
                                        all_measurement_proportions=PROPORTIONS, 
                                        current_guess=guess, 
                                        trials_per_iteration=TRIALS)
        
        GUESSES.append(new_guess)

# Save results to csv
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = Path(f"../data/netSquid_simulations/{timestamp}_qubits_{iterations*trials_per_iteration}")
results_dir.mkdir(parents=True, exist_ok=True)

# Save the protocols, proportions, and guesses each to their own csv file
with open(results_dir / "protocols.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(PROTOCOLS)
with open(results_dir / "proportions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(PROPORTIONS)
with open(results_dir / "guesses.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(GUESSES)


Iteration 0/100
Iteration 10/100
Iteration 20/100
Iteration 30/100
Iteration 40/100
Iteration 50/100
Iteration 60/100
Iteration 70/100
Iteration 80/100
Iteration 90/100
